# CME Futures: Transaction-Cost Sensitivity

For each return horizon, this notebook selects the highest validation Sharpe from the immutable
union of equal-weight signal and allocation results. It then applies the declared all-in cost grid
to that fixed configuration. Commission and slippage each receive half of the grid value.

Cost sensitivity is not a selection stage. Its rows are excluded from the final selection pool.
Contract multipliers, tick sizes, margin rates, front-contract position, roll adjustment, and
product identity remain unchanged across the grid.

In [1]:
"""Run CME futures transaction-cost sensitivity on fixed configurations."""

from case_studies.cme_futures.research_workflow import (
    ALL_LABELS,
    open_study,
    pre_overlay_candidate_set,
    product_universe_table,
    run_official_backtest_requests,
    strategy_request_frame,
)
from case_studies.utils.sweep_config import get_cost_grid_bps

## Fixed per-label inputs

The union set fails to open if either the signal or allocation population is missing or partial.
The selected backtest supplies its exact prediction checkpoint, signal, and allocation settings.

In [2]:
study = open_study(execution_tier="canonical")
universe = product_universe_table()
universe

sector,product,expiry_rule,contract_months
str,str,str,str
"""agriculture""","""ZC""","""business_day_before_15th""","""H,K,N,U,Z"""
"""agriculture""","""ZL""","""business_day_before_15th""","""F,H,K,N,Q,U,V,Z"""
"""agriculture""","""ZM""","""business_day_before_15th""","""F,H,K,N,Q,U,V,Z"""
"""agriculture""","""ZS""","""business_day_before_15th""","""F,H,K,N,Q,U,X"""
"""agriculture""","""ZW""","""business_day_before_15th""","""H,K,N,U,Z"""
…,…,…,…
"""metals""","""SI""","""3rd_last_business_day""","""H,K,N,U,Z"""
"""treasuries""","""ZB""","""last_business_day""","""H,M,U,Z"""
"""treasuries""","""ZF""","""last_business_day""","""H,M,U,Z"""


In [3]:
cost_grid = get_cost_grid_bps("cme_futures")
if not cost_grid:
    raise ValueError("the configured cost grid is empty")

request_rows = []
for label in ALL_LABELS:
    selected = pre_overlay_candidate_set(study, label=label).best_validation_sharpe()
    strategy = selected.spec()["strategy"]
    prediction_hash = selected.registry_record()["prediction_hash"]
    for total_cost_bps in cost_grid:
        request_rows.append(
            {
                "request_name": f"{selected.hash}-cost-{total_cost_bps:g}",
                "prediction_hash": prediction_hash,
                "label": label,
                "signal": strategy["signal"],
                "allocation": strategy.get("allocation"),
                "risk": None,
                "costs": {
                    "commission_bps": total_cost_bps / 2,
                    "slippage_bps": total_cost_bps / 2,
                },
                "chapter": "ch18",
            }
        )
requests = strategy_request_frame(request_rows)
requests.select("request_name", "prediction_hash", "label", "costs")

request_name,prediction_hash,label,costs
str,str,str,object
"""9bc90383acb6-cost-0""","""28e50f1ddb6d""","""fwd_ret_5d""","{'commission_bps': 0.0, 'slippage_bps': 0.0}"
"""9bc90383acb6-cost-1""","""28e50f1ddb6d""","""fwd_ret_5d""","{'commission_bps': 0.5, 'slippage_bps': 0.5}"
"""9bc90383acb6-cost-2""","""28e50f1ddb6d""","""fwd_ret_5d""","{'commission_bps': 1.0, 'slippage_bps': 1.0}"
"""9bc90383acb6-cost-3""","""28e50f1ddb6d""","""fwd_ret_5d""","{'commission_bps': 1.5, 'slippage_bps': 1.5}"
"""9bc90383acb6-cost-5""","""28e50f1ddb6d""","""fwd_ret_5d""","{'commission_bps': 2.5, 'slippage_bps': 2.5}"
…,…,…,…
"""48200fc5abeb-cost-10""","""206874caf483""","""fwd_ret_21d""","{'commission_bps': 5.0, 'slippage_bps': 5.0}"
"""48200fc5abeb-cost-15""","""206874caf483""","""fwd_ret_21d""","{'commission_bps': 7.5, 'slippage_bps': 7.5}"
"""48200fc5abeb-cost-20""","""206874caf483""","""fwd_ret_21d""","{'commission_bps': 10.0, 'slippage_bps': 10.0}"


## Execute the declared grid

Expected identities are snapshotted before execution. An empty input, failed grid member, missing
sidecar, or incomplete lineage fails the notebook instead of reporting a smaller population.

In [4]:
execution = run_official_backtest_requests(
    study,
    requests,
    population_name="cme_futures-cost-validation-v1",
)

In [5]:
execution.catalog_rows.sort("label", "request_name")

request_name,label,prediction_hash,decision_hash,backtest_hash,complete
str,str,str,str,str,bool
"""48200fc5abeb-cost-0""","""fwd_ret_21d""","""206874caf483""","""67519cb007f1""","""6748dc719115""",true
"""48200fc5abeb-cost-1""","""fwd_ret_21d""","""206874caf483""","""4ccf7b62b9f7""","""1b773b08bca3""",true
"""48200fc5abeb-cost-10""","""fwd_ret_21d""","""206874caf483""","""43dbb60ffafc""","""8e8af34b193d""",true
"""48200fc5abeb-cost-15""","""fwd_ret_21d""","""206874caf483""","""450aa33c69c1""","""f9e74352dd56""",true
"""48200fc5abeb-cost-2""","""fwd_ret_21d""","""206874caf483""","""66c19fe76a1d""","""f26771e0dd9d""",true
…,…,…,…,…,…
"""9bc90383acb6-cost-3""","""fwd_ret_5d""","""28e50f1ddb6d""","""8fa5f6bede67""","""742784d7ec57""",true
"""9bc90383acb6-cost-30""","""fwd_ret_5d""","""28e50f1ddb6d""","""574816b768b6""","""cf53a2bc600e""",true
"""9bc90383acb6-cost-5""","""fwd_ret_5d""","""28e50f1ddb6d""","""a992b7bfccac""","""d6ce2dc8c64c""",true


`17_strategy_analysis` may describe the cost curve, but these backtests do not participate in
configuration selection.